##### Loading Dependencies and  Datasets

In [1]:
import numpy as np
import pandas as pd

# Load raw datasets
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

print(f"Initial Train Shape: {train.shape}")
print(f"Initial Test Shape:  {test.shape}")

Initial Train Shape: (6818, 13)
Initial Test Shape:  (1705, 12)


In [9]:
train.head()

,id,product_code,product_weight_kg,fat_content,shelf_visibility,product_category,product_price,store_code,store_age_years,store_size,store_location_tier,store_format,total_sales
0,row_00000,PRD-PRFP9S,14.252,Low Fat,0.0271,Frozen Foods,81.37,STORE-AGY,45,Large,Tier_3,Standard Supermarket,1764.98
1,row_00001,PRD-PXXK71,7.698,Low Fat,0.0720,HEALTH AND HYGIENE,42.05,STORE-YLW,35,Small,Tier_1,Standard Supermarket,342.13
2,row_00002,PRD-V5MOIJ,14.264,Regular,0.0421,Canned,41.35,STORE-89Z,33,Medium,Tier_1,Standard Supermarket,378.85
3,row_00003,PRD-UN5Z3J,NaN,Regular,0.0449,soft drinks,174.35,STORE-7WS,47,Medium,Tier_3,Flagship Hypermarket,5595.72
4,row_00004,PRD-6RDQYB,10.338,Regular,0.0120,meat,203.06,STORE-9RG,28,Small,Tier_2,Standard Supermarket,2375.36


In [12]:
train.isnull().sum()

id                        0
product_code              0
product_weight_kg      1225
fat_content               0
shelf_visibility          0
product_category          0
product_price             0
store_code                0
store_age_years           0
store_size             1919
store_location_tier       0
store_format              0
total_sales               0
dtype: int64

In [10]:
test.head()

,id,product_code,product_weight_kg,fat_content,shelf_visibility,product_category,product_price,store_code,store_age_years,store_size,store_location_tier,store_format
0,row_00009,PRD-2WLNC8,8.628,Low Fat,0.0167,household,190.72,STORE-HL7,23,Medium,Tier_3,Superstore
1,row_00015,PRD-LV9PLM,6.993,Low Fat,0.0579,Health and Hygiene,261.07,STORE-9RG,28,Small,Tier_2,Standard Supermarket
2,row_00019,PRD-ET6ZI7,NaN,Low Fat,0.1262,FRUITS AND VEGETABLES,112.50,STORE-7WS,47,Medium,Tier_3,Flagship Hypermarket
3,row_00020,PRD-6RX35U,14.034,Regular,0.0761,frozen foods,200.71,STORE-DKU,30,NaN,Tier_2,Standard Supermarket
4,row_00023,PRD-I4J38V,13.063,Low Fat,0.0650,Frozen Foods,150.59,STORE-89Z,33,Medium,Tier_1,Standard Supermarket


In [13]:
test.isnull().sum()

id                       0
product_code             0
product_weight_kg      306
fat_content              0
shelf_visibility         0
product_category         0
product_price            0
store_code               0
store_age_years          0
store_size             491
store_location_tier      0
store_format             0
dtype: int64

###### Datasets info

In [2]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6818 entries, 0 to 6817
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   6818 non-null   object 
 1   product_code         6818 non-null   object 
 2   product_weight_kg    5593 non-null   float64
 3   fat_content          6818 non-null   object 
 4   shelf_visibility     6818 non-null   float64
 5   product_category     6818 non-null   object 
 6   product_price        6818 non-null   float64
 7   store_code           6818 non-null   object 
 8   store_age_years      6818 non-null   int64  
 9   store_size           4899 non-null   object 
 10  store_location_tier  6818 non-null   object 
 11  store_format         6818 non-null   object 
 12  total_sales          6818 non-null   float64
dtypes: float64(4), int64(1), object(8)
memory usage: 692.6+ KB


In [3]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1705 entries, 0 to 1704
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   1705 non-null   object 
 1   product_code         1705 non-null   object 
 2   product_weight_kg    1399 non-null   float64
 3   fat_content          1705 non-null   object 
 4   shelf_visibility     1705 non-null   float64
 5   product_category     1705 non-null   object 
 6   product_price        1705 non-null   float64
 7   store_code           1705 non-null   object 
 8   store_age_years      1705 non-null   int64  
 9   store_size           1214 non-null   object 
 10  store_location_tier  1705 non-null   object 
 11  store_format         1705 non-null   object 
dtypes: float64(3), int64(1), object(8)
memory usage: 160.0+ KB


###### Data Cleaning

In [4]:
# Combining datasets temporarily to ensure consistent cleaning across train & test
target_col = 'total_sales'
train_len = len(train)

if target_col in train.columns:
    target = train[target_col]
    df = pd.concat([train.drop(columns=[target_col]), test], ignore_index=True)
else:
    df = pd.concat([train, test], ignore_index=True)

In [5]:
# Cleaning product_category capitalization
df['product_category'] = df['product_category'].astype(str).str.title().str.strip()

# Standardizing fat_content variants
fat_mapping = {
    'low fat': 'Low Fat',
    'lf': 'Low Fat',
    'lowfat': 'Low Fat',
    'low_fat': 'Low Fat',
    'regular': 'Regular',
    'reg': 'Regular'
}
df['fat_content'] = (
    df['fat_content']
    .astype(str)
    .str.lower()
    .str.strip()
    .map(fat_mapping)
    .fillna('Regular')
)

In [6]:
# Removing extra spaces
string_cols = ['product_code', 'store_code', 'store_location_tier', 'store_format']
for col in string_cols:
    df[col] = df[col].astype(str).str.strip()

In [7]:
# Handling 0 shelf_visibility anomaly
df['shelf_visibility'] = df['shelf_visibility'].replace(0, np.nan)
df['shelf_visibility'] = (
    df.groupby('product_code')['shelf_visibility']
    .transform(lambda x: x.fillna(x.mean()))
)
df['shelf_visibility'] = df['shelf_visibility'].fillna(df['shelf_visibility'].mean())

# Impute product_weight_kg 
df['product_weight_kg'] = (
    df.groupby('product_code')['product_weight_kg']
    .transform(lambda x: x.fillna(x.median()))
)
df['product_weight_kg'] = df['product_weight_kg'].fillna(df['product_weight_kg'].median())

# Impute store_size 
store_size_mode = df.groupby('store_format')['store_size'].transform(
    lambda x: x.mode()[0] if not x.mode().empty else 'Medium'
)
df['store_size'] = df['store_size'].fillna(store_size_mode)

In [8]:
df.head()

,id,product_code,product_weight_kg,fat_content,shelf_visibility,product_category,product_price,store_code,store_age_years,store_size,store_location_tier,store_format
0,row_00000,PRD-PRFP9S,14.252,Low Fat,0.0271,Frozen Foods,81.37,STORE-AGY,45,Large,Tier_3,Standard Supermarket
1,row_00001,PRD-PXXK71,7.698,Low Fat,0.0720,Health And Hygiene,42.05,STORE-YLW,35,Small,Tier_1,Standard Supermarket
2,row_00002,PRD-V5MOIJ,14.264,Regular,0.0421,Canned,41.35,STORE-89Z,33,Medium,Tier_1,Standard Supermarket
3,row_00003,PRD-UN5Z3J,12.665,Regular,0.0449,Soft Drinks,174.35,STORE-7WS,47,Medium,Tier_3,Flagship Hypermarket
4,row_00004,PRD-6RDQYB,10.338,Regular,0.0120,Meat,203.06,STORE-9RG,28,Small,Tier_2,Standard Supermarket


In [11]:
df.isnull().sum()

id                     0
product_code           0
product_weight_kg      0
fat_content            0
shelf_visibility       0
product_category       0
product_price          0
store_code             0
store_age_years        0
store_size             0
store_location_tier    0
store_format           0
dtype: int64

In [14]:
# Separating datasets
train_clean = df.iloc[:train_len].copy()
test_clean = df.iloc[train_len:].copy()

# Adding target column back to train_clean
train_clean[target_col] = target.values

# Verifying no missing values remain
print("TRAIN CLEAN MISSING VALUES")
print(train_clean.isnull().sum())

print("\nTEST CLEAN MISSING VALUES")
print(test_clean.isnull().sum())

--- TRAIN CLEAN MISSING VALUES ---
id                     0
product_code           0
product_weight_kg      0
fat_content            0
shelf_visibility       0
product_category       0
product_price          0
store_code             0
store_age_years        0
store_size             0
store_location_tier    0
store_format           0
total_sales            0
dtype: int64

--- TEST CLEAN MISSING VALUES ---
id                     0
product_code           0
product_weight_kg      0
fat_content            0
shelf_visibility       0
product_category       0
product_price          0
store_code             0
store_age_years        0
store_size             0
store_location_tier    0
store_format           0
dtype: int64


In [15]:
# Saving clean datasets to data folder
train_clean.to_csv('data/train_cleaned.csv', index=False)
test_clean.to_csv('data/test_cleaned.csv', index=False)

print("\nSaved as 'data/train_cleaned.csv' and 'data/test_cleaned.csv'.")


Cleaning complete! Saved as 'data/train_cleaned.csv' and 'data/test_cleaned.csv'.
